# Stage 2 — NER Pipeline
**Goal:** Extract person names from every manifesto using CamemBERT-NER.

By the end of this notebook you will have:
- A citation dataset: every person name mentioned in every manifesto
- Each citation linked to the candidate's gender, year, and party
- A quick evaluation of the NER quality on a sample

## 0. Setup

In [ ]:
import pandas as pd
from pathlib import Path
from transformers import pipeline
from transformers import AutoConfig
from tqdm import tqdm
from transformers import AutoTokenizer

PROJECT_ROOT = Path("..")

df = pd.read_csv(PROJECT_ROOT / "data" / "manifestos_with_metadata.csv", sep=",", encoding="utf-8", low_memory=False)  

print(f"Manifestos to process: {len(df):,}")
print(df["titulaire-sexe"].value_counts())

Manifestos to process: 20,331
titulaire-sexe
homme    18171
femme     2160
Name: count, dtype: int64


In [2]:
df.head(3)

,year_x,candidate_id,text,n_words,id,titulaire-sexe,titulaire-soutien,titulaire-nom,titulaire-prenom,titulaire-profession,departement-nom,year_y
0,1973,EL065_L_1973_03_001_01_1_PF_01,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,492,EL065_L_1973_03_001_01_1_PF_01,homme,Centre progrès et démocratie moderne;Union des...,Barberot,Paul,non mentionné,Ain,1973
1,1973,EL065_L_1973_03_001_01_1_PF_02,REPUBLIQUE FRANCAISE - LIBERTE - EGALITE - FRA...,828,EL065_L_1973_03_001_01_1_PF_02,homme,Parti socialiste;Mouvement des radicaux de gauche,Monnet,Roland,ingénieur EDF,Ain,1973
2,1973,EL065_L_1973_03_001_01_1_PF_03,Sciences Po / fonds CEVIPOF\nREPUBLIQUE FRANÇA...,390,EL065_L_1973_03_001_01_1_PF_03,homme,non mentionné,Chomarat,Michel,cadre technique,Ain,1973


## 1. Load CamemBERT-NER

`Jean-Baptiste/camembert-ner` is a CamemBERT fine-tune model trained on French NER datasets (WikiNER + French TreeBank).
It recognizes 4 entity types: **PER** (person), LOC (location), ORG (organization), MISC (miscellaneous).

For this project, I am interested in retrieving **PER** — the person names mentioned in manifestos.

In [3]:
ner = pipeline(
    "ner",
    model="Jean-Baptiste/camembert-ner",
    aggregation_strategy="simple",  # merges token pieces into full entity spans
    device= "mps" #0 #-1  
)
print("Model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CamembertForTokenClassification LOAD REPORT from: Jean-Baptiste/camembert-ner
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


## 2. Run NER on manifestos

CamemBERT-NER has a hard limit of 512 tokens. The median manifesto in our corpus is 963 tokens, and 88% exceed this limit, meaning NER in Stage 2 only sees the first ~350 words of most texts. Citations appearing later in longer manifestos will be missed. I accept this constraint given computational limits; candidates typically name key political figures early in their manifesto.

In [5]:
sample_text = df["text"].iloc[0]
sample_candidate = df["candidate_id"].iloc[0]
print(f"Candidate: {sample_candidate}")
print(f"Text preview: {sample_text[:300]}")
print("\n--- NER output ---")

entities = ner(sample_text[:1000])  # test on first 1000 chars
persons = [e for e in entities if e["entity_group"] == "PER"]
for p in persons:
    print(f"  {p['word']!r:30s}  score={p['score']:.2f}")

Candidate: EL065_L_1973_03_001_01_1_PF_01
Text preview: Sciences Po / fonds CEVIPOF
REPUBLIQUE FRANÇAISE - LIBERTE - EGALITE - FRATERNITE
Paul BARBEROT
Département de l'AIN 1re Circonscription (BOURG-EN-BRESSE)
ÉLECTIONS LÉGISLATIVES du 4 MARS 1973
CENTRE - PROGRES ET DEMOCRATIE MODERNE
investi par L'UNION DES RÉPUBLICAINS DE PROGRÈS pour le soutien au P

--- NER output ---
  'Paul BARBEROT'                 score=0.99
  'Président de la République'    score=0.75


In [ ]:
config = AutoConfig.from_pretrained("Jean-Baptiste/camembert-ner")                            
print(config.max_position_embeddings)

514


In [10]:
tokenizer = AutoTokenizer.from_pretrained("Jean-Baptiste/camembert-ner")                                                                                    
                                                                                                                                                              
df["n_tokens"] = df["text"].apply(                                                                                                    
    lambda t: len(tokenizer.encode(t, add_special_tokens=False))                                                                                            
  )               
                                                                                                                                                              
print("Token count summary:")           
print(df["n_tokens"].describe().round(0))                                                                                                        
print(f"\nManifestos exceeding 512 tokens: {(df['n_tokens'] > 512).sum():,} ({(df['n_tokens'] > 512).mean()*100:.1f}%)")

Token count summary:
count    20331.0
mean      1082.0
std        584.0
min         48.0
25%        668.0
50%        963.0
75%       1395.0
max       7960.0
Name: n_tokens, dtype: float64

Manifestos exceeding 512 tokens: 17,887 (88.0%)


In [6]:
def extract_persons(text, max_chars=2000):
    if not text or len(text.strip()) == 0:
        return []
    entities = ner(text[:max_chars])
    return [e["word"] for e in entities if e["entity_group"] == "PER" and e["score"] > 0.7]

citations = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Running NER"):
    persons = extract_persons(row["text"])
    year = row["year_x"] if "year_x" in row.index else row["year"]
    for person in persons:
        citations.append({
            "candidate_id":    row["candidate_id"],
            "candidate_sex":   row["titulaire-sexe"],
            "candidate_party": row["titulaire-soutien"],
            "candidate_name":  f"{row['titulaire-prenom']} {row['titulaire-nom']}",
            "year":            year,
            "cited_person":    person,
        })

df_citations = pd.DataFrame(citations)
print(f"\nTotal citations extracted: {len(df_citations):,}")
print(f"Unique persons cited: {df_citations['cited_person'].nunique():,}")
df_citations.head(10)

Running NER:   0%|          | 41/20331 [00:05<41:38,  8.12it/s]  


KeyboardInterrupt: 

In [ ]:
out_path = PROJECT_ROOT / "data" / "citations.csv"
df_citations.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to ../data/citations.csv


## 3. Clean & Normalize Citations

Two problems to fix before the gender analysis:
1. **Duplicate names** — same person cited in different formats (`François MITTERRAND`, `Mitterrand`, `MITTERRAND`)
2. **Non-person entities** — titles and honorifics that slipped through NER (`Monsieur`, `Président`, `Mademoiselle`)

**Strategy:**
- Remove known noise tokens (titles, honorifics, common words)
- Normalize casing: convert to Title Case
- Remove very short strings (single words under 3 chars)
- Keep only citations appearing at least twice overall (very rare = likely noise)

In [ ]:
import re
import unicodedata

citations_path = PROJECT_ROOT / "data" / "citations.csv"
df_citations = pd.read_csv(citations_path, sep=",", encoding="utf-8", low_memory=False)

def strip_accents(s):
    s = unicodedata.normalize("NFD", s)
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

def normalize_name(name):
    name = " ".join(name.strip().split()).title()
    return strip_accents(name)

# Noise tokens normalized at definition time so they match accent-stripped names
_RAW_NOISE = {
    "monsieur", "madame", "mademoiselle", "mme", "m.",
    "président", "presidente", "president",
    "président de la république", "premier ministre",
    "le président de la république", "le premier ministre",
    "général", "general", "général de gaulle", "general de gaulle",
    "monseigneur", "docteur", "dr", "professeur", "pr",
    "cher monsieur", "chère madame", "cher ami",
}
NOISE_TOKENS = {strip_accents(t) for t in _RAW_NOISE}

def is_noise(name_normalized):
    return name_normalized.lower() in NOISE_TOKENS

def has_enough_tokens(name_normalized, min_tokens=2):
    return len(name_normalized.split()) >= min_tokens

def has_truncated_token(name_normalized):
    tokens = name_normalized.split()
    return len(tokens[-1]) == 1

df_citations["cited_person_clean"] = df_citations["cited_person"].apply(normalize_name)

mask_noise = df_citations["cited_person_clean"].apply(is_noise)
print(f"Noise entries removed: {mask_noise.sum():,}")
df_clean = df_citations[~mask_noise].copy()

mask_short = ~df_clean["cited_person_clean"].apply(has_enough_tokens)
print(f"Single-word entries removed: {mask_short.sum():,}")
df_clean = df_clean[~mask_short].copy()

mask_truncated = df_clean["cited_person_clean"].apply(has_truncated_token)
print(f"Truncated-name entries removed: {mask_truncated.sum():,}")
df_clean = df_clean[~mask_truncated].copy()

counts = df_clean["cited_person_clean"].value_counts()
freq_names = counts[counts >= 2].index
mask_rare = ~df_clean["cited_person_clean"].isin(freq_names)
print(f"Rare (< 2 occurrences) entries removed: {mask_rare.sum():,}")
df_clean = df_clean[~mask_rare].copy()

print(f"\nCitations after cleaning: {len(df_clean):,}  (was {len(df_citations):,})")
print(f"Unique persons after cleaning: {df_clean['cited_person_clean'].nunique():,}")

print("\n=== Top 20 after cleaning ===")
print(df_clean["cited_person_clean"].value_counts().head(20).to_string())

out_path_clean = PROJECT_ROOT / "data" / "citations_clean.csv"
df_clean.to_csv(out_path_clean, index=False)
print(f"\nSaved to {out_path_clean}")

Noise entries removed: 4,910
Single-word entries removed: 10,470
Truncated-name entries removed: 441
Rare (< 2 occurrences) entries removed: 19,590

Citations after cleaning: 50,416  (was 85,827)
Unique persons after cleaning: 10,139

=== Top 20 after cleaning ===
cited_person_clean
Francois Mitterrand         3725
Michel Rocard                841
Arlette Laguiller            674
Le Pen                       544
Jean-Marie Le Pen            530
Jacques Chirac               447
Valery Giscard D'Estaing     366
Brice Lalonde                322
Giscard D'Estaing            292
Georges Pompidou             277
Pierre Mauroy                230
Raymond Barre                218
F. Mitterrand                192
President Pompidou           163
Georges Marchais             156
Antoine Pinay                153
Robert Fabre                 143
Michel Jobert                142
Jean Lecanuet                 97
De Gaulle                     90

Saved to ../data/citations_clean.csv


The cleaning pipeline removed noise in 4 steps:

* 4,910 noise entries removed: titles and honorifics that NER misclassified as persons: "Monsieur", "Madame", "Président de la République", "Général de Gaulle", etc.                           

* 10,470 single-word entries removed.         

* 441 truncated-name entries removed: artifacts like "Giscard D" where the name got cut mid-word due to OCR or the 2000-character truncation.                
                                          
* 19,590 rare entries removed: names appearing only once in the entire corpus, almost certainly OCR noise or very minor local figures with no analytical value.             


Result: from 85,827 raw citations down to 50,416 clean ones, with 10,139 unique person names. The top 20 are all real, historically recognizable French politicians from 1973–1993.

## 4. Analysis of top cited figures

Before moving to the gender analysis, sanity-check the results:
are the most cited figures actual politicians? Do they make historical sense?

Across all 20,331 manifestos, François Mitterrand is the most cited political figure. Other dominant figures include Michel Rocard, Jean-Marie Le Pen, Arlette Laguiller, and the centrist figures Giscard d'Estaing and Pompidou.                                       
                                                                                                                                                              
When broken down by candidate gender I observe differences in the gender of the people cited: male candidates overwhelmingly cite mainstream political figures: Chirac, Giscard d'Estaing, Pompidou — the established male political elite. 
Female candidates, while also citing Mitterrand and Le Pen, show a strikingly different pattern: Gisèle Halimi (feminist lawyer), Simone de Beauvoir (feminist philosopher), Jean Rostand and Jacques Monod (humanist scientists) appear prominently in female manifestos but are virtually absent from male ones. This suggests that female candidates drew on a distinct intellectual and feminist tradition when writing their manifestos — one that male candidates largely ignored.

In [15]:
df_clean = pd.read_csv(PROJECT_ROOT / "data" / "citations_clean.csv", sep=",", encoding="utf-8", low_memory=False) 
print("=== Top 30 most cited persons overall ===")
print(df_clean["cited_person_clean"].value_counts().head(30).to_string())

=== Top 30 most cited persons overall ===
cited_person_clean
Francois Mitterrand               3730
François Mitterrand               3707
Michel Rocard                     1257
Arlette Laguiller                  840
Le Pen                             693
Jean-Marie Le Pen                  665
Jacques Chirac                     538
Giscard D'Estaing                  415
Brice Lalonde                      393
Georges Pompidou                   376
Valery Giscard D'Estaing           369
Pierre Mauroy                      362
Valéry Giscard D'Estaing           339
Raymond Barre                      287
F. Mitterrand                      241
Georges Marchais                   218
Robert Fabre                       194
Antoine Pinay                      174
Michel Jobert                      169
President Pompidou                 163
Président Pompidou                 162
Huguette Bouchardeau               124
De Gaulle                          113
Madame, Mademoiselle, Monsieur     108
Jea

In [16]:
print("=== Top 20 cited by FEMALE candidates ===")
female_cit = df_clean[df_clean["candidate_sex"] == "femme"]
print(female_cit["cited_person_clean"].value_counts().head(20).to_string())

print("\n=== Top 20 cited by MALE candidates ===")
male_cit = df_clean[df_clean["candidate_sex"] == "homme"]
print(male_cit["cited_person_clean"].value_counts().head(20).to_string())

=== Top 20 cited by FEMALE candidates ===
cited_person_clean
Francois Mitterrand     447
François Mitterrand     443
Arlette Laguiller       331
Michel Rocard           145
Jean-Marie Le Pen        97
Le Pen                   93
Gisele Halimi            90
Brice Lalonde            89
De Wendel                76
Simone De Beauvoir       57
Jean Rostand             48
Jacques Monod            48
Jacques Chirac           43
Giscard D'Estaing        43
Gisèle Halimi            41
Georges Pompidou         32
Huguette Bouchardeau     29
F. Mitterrand            25
Président Pompidou       24
President Pompidou       24

=== Top 20 cited by MALE candidates ===
cited_person_clean
Francois Mitterrand         3283
François Mitterrand         3264
Michel Rocard               1112
Le Pen                       600
Jean-Marie Le Pen            568
Arlette Laguiller            509
Jacques Chirac               495
Giscard D'Estaing            372
Valery Giscard D'Estaing     350
Georges Pompidou      

In [14]:
print("=== Citations per year ===")
year_col = "year_x" if "year_x" in df_clean.columns else "year"
print(df_clean.groupby([year_col, "candidate_sex"]).size().unstack(fill_value=0))

=== Citations per year ===
candidate_sex  femme  homme
year                       
1973.0           396   9334
1978.0          1599  11805
1981.0          1542  14273
1988.0          1767  16499
1993.0          1338  12357
